<a href="https://colab.research.google.com/github/kjahan/nlp_tips_and_trick/blob/main/notebooks/cross_encoders.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cross Encoder vs Bi-Encoder

Nils talks about a few architecture for training Dense model for retreival tasks in the presence of unkown keywords or out of domain distribution!

https://www.youtube.com/watch?v=xbdLowiQTlk

## When to use Cross- / Bi-Encoders?

Cross-Encoders can be used whenever you have a pre-defined set of sentence pairs you want to score. For example, you have 100 sentence pairs and you want to get similarity scores for these 100 pairs.

Bi-Encoders (see Computing Sentence Embeddings) are used whenever you need a sentence embedding in a vector space for efficient comparison. Applications are for example Information Retrieval / Semantic Search or Clustering. Cross-Encoders would be the wrong choice for these application: Clustering 10,000 sentence with CrossEncoders would require computing similarity scores for about 50 Million sentence combinations, which takes about 65 hours. With a Bi-Encoder, you compute the embedding for each sentence, which takes only 5 seconds. You can then perform the clustering.

## Cross-Encoders Usage
Using Cross-Encoders is quite easy:

https://www.sbert.net/examples/applications/cross-encoder/README.html

In [1]:
!pip install -U sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 931.3 kB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 3.0 MB/s eta 0:00:00
  Created wheel for sentence-transformers: filename=sentence_transformers-2.2.2-py3-none-any.whl size=125923 sha256=7c722a10474c0850edd05421417d4ec0ecc35b0f3c0886fde0b165364a6a3ca8
  Stored in directory: /root/.cache/pip/wheels/62/f2/10/1e606fd5f02395388f74e7462910fe851042f97238cbbd902f
Successfully built sentence-transformers


In [16]:
from sentence_transformers.cross_encoder import CrossEncoder
from sentence_transformers import SentenceTransformer
import numpy as np

## Cross Encoder models

https://www.sbert.net/docs/pretrained_cross-encoders.html

MS MARCO Passage Retrieval is a large dataset with real user queries from Bing search engine with annotated relevant text passages.



`cross-encoder/ms-marco-TinyBERT-L-2-v2 - MRR@10 on MS Marco Dev Set: 32.56`

`cross-encoder/ms-marco-MiniLM-L-2-v2 - MRR@10 on MS Marco Dev Set: 34.85`

`cross-encoder/ms-marco-MiniLM-L-4-v2 - MRR@10 on MS Marco Dev Set: 37.70`

`cross-encoder/ms-marco-MiniLM-L-6-v2 - MRR@10 on MS Marco Dev Set: 39.01`

`cross-encoder/ms-marco-MiniLM-L-12-v2 - MRR@10 on MS Marco Dev Set: 39.02`

In [5]:
model_msmacro = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-12-v2', max_length=512)

config.json:   0%|          | 0.00/791 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/134M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [7]:
scores = model_msmacro.predict([["corona virus", "covid-19"],
                        ["corona virus", "sars-cov-2"]])

In [8]:
scores

array([-10.650173,  -9.583521], dtype=float32)

In [9]:
scores = model_msmacro.predict([["usa", "united states"],
                        ["usa", "america"]])

In [10]:
scores

array([1.426301 , 0.6048379], dtype=float32)

In [11]:
scores = model_msmacro.predict([["how many people live in usa?", "what is the population of usa?"],
                        ["what is the capital of usa?", "What is USA capital city?"]])

In [12]:
scores

array([0.51250845, 4.690867  ], dtype=float32)

## Bi-Encoder

In [14]:
model_sbert = SentenceTransformer('all-mpnet-base-v2')

.gitattributes:   0%|          | 0.00/1.18k [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.6k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

train_script.py:   0%|          | 0.00/13.1k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

In [15]:
#Our sentences we like to encode
sentences = ["corona virus", "covid-19", "sars-cov-2"]

#Sentences are encoded by calling model.encode()
embeddings = model_sbert.encode(sentences)

In [18]:
tot_sim_1 = np.dot(embeddings[0], embeddings[1])
tot_sim_2 = np.dot(embeddings[1], embeddings[2])

print("tot_sim_1: {}".format(tot_sim_1))
print("tot_sim_2: {}".format(tot_sim_2))

tot_sim_1: 0.7194761037826538
tot_sim_2: 0.7089287042617798


In [21]:
sentences = ["what is corona virus?", "Coronavirus is a type of virus."]

embeddings = model_sbert.encode(sentences)
tot_sim_1 = np.dot(embeddings[0], embeddings[1])

print("tot_sim_1: {}".format(tot_sim_1))

tot_sim_1: 0.8249198198318481


In [22]:
sentences = ["what is covid-19?", "Coronavirus is a type of virus."]

embeddings = model_sbert.encode(sentences)
tot_sim_1 = np.dot(embeddings[0], embeddings[1])

print("tot_sim_1: {}".format(tot_sim_1))

tot_sim_1: 0.5878921747207642


In [23]:
sentences = ["what is covid-19?", "covid-19 is a type of virus."]

embeddings = model_sbert.encode(sentences)
tot_sim_1 = np.dot(embeddings[0], embeddings[1])

print("tot_sim_1: {}".format(tot_sim_1))

tot_sim_1: 0.8282651901245117


In [24]:
scores = model_msmacro.predict([["what is corona virus?", "Coronavirus is a type of virus."],
                        ["what is covid-19?", "Coronavirus is a type of virus."]])

scores

array([  9.547815, -11.092532], dtype=float32)

## SQuAD (QNLI)

QNLI is based on the SQuAD dataset and was introduced by the GLUE Benchmark. Given a passage from Wikipedia, annotators created questions that are answerable by that passage.

`cross-encoder/qnli-distilroberta-base - Accuracy on QNLI dev set: 90.96`

`cross-encoder/qnli-electra-base - Accuracy on QNLI dev set: 93.21`

In [25]:
model_qnli = CrossEncoder('cross-encoder/qnli-electra-base', max_length=512)

config.json:   0%|          | 0.00/771 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/268 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

## Quora Duplicate Questions
These models have been trained on the Quora duplicate questions dataset. They can used like the STSb models and give a score 0…1 indicating the probability that two questions are duplicate questions.

`cross-encoder/quora-distilroberta-base - Average Precision dev set: 87.48`

`cross-encoder/quora-roberta-base - Average Precision dev set: 87.80`

`cross-encoder/quora-roberta-large - Average Precision dev set: 87.91`

In [28]:
model_quora = CrossEncoder('cross-encoder/quora-roberta-large', max_length=512)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/139 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

# NLI

Given two sentences, are these contradicting each other, entailing one the other or are these netural? The following models were trained on the SNLI and MultiNLI datasets.

`cross-encoder/nli-deberta-v3-base - Accuracy on MNLI mismatched set: 90.04`

`cross-encoder/nli-deberta-base - Accuracy on MNLI mismatched set: 88.08`

`cross-encoder/nli-deberta-v3-xsmall - Accuracy on MNLI mismatched set: 87.77`

`cross-encoder/nli-deberta-v3-small - Accuracy on MNLI mismatched set: 87.55`

`cross-encoder/nli-roberta-base - Accuracy on MNLI mismatched set: 87.47`

`cross-encoder/nli-MiniLM2-L6-H768 - Accuracy on MNLI mismatched set: 86.89`

`cross-encoder/nli-distilroberta-base - Accuracy on MNLI mismatched set: 83.98`

In [26]:
model_nli = CrossEncoder('cross-encoder/nli-deberta-v3-base', max_length=512)

config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/738M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/417 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/18.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/convert_slow_tokenizer.py:473: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [27]:
scores = model_qnli.predict([["what is corona virus?", "Coronavirus is a type of virus."],
                        ["what is covid-19?", "Coronavirus is a type of virus."]])

scores

array([0.99533916, 0.00596726], dtype=float32)

In [29]:
scores = model_quora.predict([["what is corona virus?", "Coronavirus is a type of virus."],
                        ["what is covid-19?", "Coronavirus is a type of virus."]])

scores

array([0.00975777, 0.00540469], dtype=float32)

In [30]:
scores = model_nli.predict([["what is corona virus?", "Coronavirus is a type of virus."],
                        ["what is covid-19?", "Coronavirus is a type of virus."]])

scores

array([[-5.0526004 ,  4.038754  , -0.25298315],
       [-4.762154  ,  3.827151  , -0.25090843]], dtype=float32)

# Doc2query

https://huggingface.co/doc2query/msmarco-t5-base-v1

use cases:

`Document expansion: You generate for your paragraphs 20-40 queries and index the paragraphs and the generates queries in a standard BM25 index like Elasticsearch, OpenSearch, or Lucene. The generated queries help to close the lexical gap of lexical search, as the generate queries contain synonyms. Further, it re-weights words giving important words a higher weight even if they appear seldomn in a paragraph. In our BEIR paper we showed that BM25+docT5query is a powerful search engine. In the BEIR repository we have an example how to use docT5query with Pyserini.`

`Domain Specific Training Data Generation: It can be used to generate training data to learn an embedding model. On SBERT.net we have an example how to use the model to generate (query, text) pairs for a given collection of unlabeled texts. These pairs can then be used to train powerful dense embedding models.`

In [31]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

model_name = 'doc2query/msmarco-t5-base-v1'
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/2.12k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.79k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thouroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


config.json:   0%|          | 0.00/667 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

In [32]:
text = "Python is an interpreted, high-level and general-purpose programming language. Python's design philosophy emphasizes code readability with its notable use of significant whitespace. Its language constructs and object-oriented approach aim to help programmers write clear, logical code for small and large-scale projects."

## Generation

Note: `model.generate()` is non-deterministic. It produces different queries each time you run it.

In [33]:
input_ids = tokenizer.encode(text, max_length=320, truncation=True, return_tensors='pt')
outputs = model.generate(
    input_ids=input_ids,
    max_length=64,
    do_sample=True,
    top_p=0.95,
    num_return_sequences=5)

In [34]:
print("Text:")
print(text)

print("\nGenerated Queries:")
for i in range(len(outputs)):
    query = tokenizer.decode(outputs[i], skip_special_tokens=True)
    print(f'{i + 1}: {query}')

Text:
Python is an interpreted, high-level and general-purpose programming language. Python's design philosophy emphasizes code readability with its notable use of significant whitespace. Its language constructs and object-oriented approach aim to help programmers write clear, logical code for small and large-scale projects.

Generated Queries:
1: what is python
2: what programming languages are available?
3: what is python language
4: what is python
5: what is python in computer terms


## Kidney stone

https://en.wikipedia.org/wiki/Kidney_stone_disease

In [35]:
text = """Kidney stone disease, also known as renal calculus disease, nephrolithiasis or urolithiasis, is a crystallopathy where a solid piece of material (renal calculus) develops in the urinary tract. Renal calculi typically form in the kidney and leave the body in the urine stream. A small calculus may pass without causing symptoms. If a stone grows to more than 5 millimeters (0.2 inches), it can cause blockage of the ureter, resulting in sharp and severe pain in the lower back or abdomen. A calculus may also result in blood in the urine, vomiting, or painful urination. About half of people who have had a renal calculus are likely to have another within ten years."""

input_ids = tokenizer.encode(text, max_length=320, truncation=True, return_tensors='pt')
outputs = model.generate(
    input_ids=input_ids,
    max_length=64,
    do_sample=True,
    top_p=0.95,
    num_return_sequences=5)

In [36]:
print("Text:")
print(text)

print("\nGenerated Queries:")
for i in range(len(outputs)):
    query = tokenizer.decode(outputs[i], skip_special_tokens=True)
    print(f'{i + 1}: {query}')

Text:
Kidney stone disease, also known as renal calculus disease, nephrolithiasis or urolithiasis, is a crystallopathy where a solid piece of material (renal calculus) develops in the urinary tract. Renal calculi typically form in the kidney and leave the body in the urine stream. A small calculus may pass without causing symptoms. If a stone grows to more than 5 millimeters (0.2 inches), it can cause blockage of the ureter, resulting in sharp and severe pain in the lower back or abdomen. A calculus may also result in blood in the urine, vomiting, or painful urination. About half of people who have had a renal calculus are likely to have another within ten years.

Generated Queries:
1: where is a kidney stone formed
2: what is kidney stones
3: what is the cause of kidney stones
4: what is stone in kidney
5: can kidney stone form in the urine


## NYT Lawsuite



In [44]:
text = """For example, in 2019, The Times published a Pulitzer-prize winning, five-part
series on predatory lending in New York City’s taxi industry. The 18-month investigation included
600 interviews, more than 100 records requests, large-scale data analysis, and the review of
thousands of pages of internal bank records and other documents, and ultimately led to criminal
probes and the enactment of new laws to prevent future abuse. OpenAI had no role in the creation
of this content, yet with minimal prompting, will recite large portions of it verbatim."""

input_ids = tokenizer.encode(text, max_length=320, truncation=True, return_tensors='pt')
outputs = model.generate(
    input_ids=input_ids,
    max_length=64,
    do_sample=True,
    top_p=0.95,
    num_return_sequences=5)

In [45]:
print("Text:")
print(text)

print("\nGenerated Queries:")
for i in range(len(outputs)):
    query = tokenizer.decode(outputs[i], skip_special_tokens=True)
    print(f'{i + 1}: {query}')

Text:
For example, in 2019, The Times published a Pulitzer-prize winning, five-part
series on predatory lending in New York City’s taxi industry. The 18-month investigation included
600 interviews, more than 100 records requests, large-scale data analysis, and the review of
thousands of pages of internal bank records and other documents, and ultimately led to criminal
probes and the enactment of new laws to prevent future abuse. OpenAI had no role in the creation
of this content, yet with minimal prompting, will recite large portions of it verbatim.

Generated Queries:
1: when was openai in nyc
2: what does openai know about taxi industry
3: psi of predatory lending in nyc
4: what was the prussian press prize winning investigation into predatory lending
5: when did openai's crime report come out


In [42]:
text = """For example, in 2019, The Times published a Pulitzer-prize winning, five-part
series on predatory lending in New York City’s taxi industry. The 18-month investigation included
600 interviews, more than 100 records requests, large-scale data analysis, and the review of
thousands of pages of internal bank records and other documents, and ultimately led to criminal
probes and the enactment of new laws to prevent future abuse."""

input_ids = tokenizer.encode(text, max_length=320, truncation=True, return_tensors='pt')
outputs = model.generate(
    input_ids=input_ids,
    max_length=64,
    do_sample=True,
    top_p=0.95,
    num_return_sequences=5)

In [43]:
print("Text:")
print(text)

print("\nGenerated Queries:")
for i in range(len(outputs)):
    query = tokenizer.decode(outputs[i], skip_special_tokens=True)
    print(f'{i + 1}: {query}')

Text:
For example, in 2019, The Times published a Pulitzer-prize winning, five-part
series on predatory lending in New York City’s taxi industry. The 18-month investigation included
600 interviews, more than 100 records requests, large-scale data analysis, and the review of
thousands of pages of internal bank records and other documents, and ultimately led to criminal
probes and the enactment of new laws to prevent future abuse.

Generated Queries:
1: what kind of money ripes a lot in new york city
2: how long did it take for a court to rule against predatory lending
3: what investigation did the times do
4: why did the times publish a scandal about predatory lending
5: where are bank records of nyc


# Domain Adaptation

The goal of Domain Adaptation is to adapt text embedding models to your specific text domain without the need to labeled training data.

https://www.sbert.net/examples/domain_adaptation/README.html

GPL paper: Pseudo Labeling using cross encoder:

https://arxiv.org/pdf/2112.07577.pdf

He talks about GPL in hide video here:

https://youtu.be/xbdLowiQTlk?si=_w8iZkXKxEDnxgDy&t=1762

Essentially they generate query from passages. Then they have positiove and also sample negative using a bi-encoder using the generated query. But some of the negative passages are actually not negative so he uses cross -encoder to assign a score and then f\do final training:

`Training: Once we have the triplets (generated query, positive passage, mined negative passage) and the Cross-Encoder socres for (query, positive) and (query, negative) we can start training the text embedding model using MarginMSELoss.`

In [48]:
query = "what is futures contract "

pos = "Futures are a type of derivative contract agreement to buy or sell a specific commodity asset or security at a set future date for a set price. Futures contracts, or simply 'futures,' are traded on futures exchanges like the CME Group and require a brokerage account that's approved to trade futures."

# Easy negatives only mention "future contract"
neg_1 = 'Anyway in this one example the s&p 500 futures contract has an "initial margin" of $19,250, meaning'

neg_2 = "but the moment you exercise you must have $5,940 in a margin account to actually use the futures contract"

# False negative
neg_3 = "a futures contract is simply a contract that requires party A to buy a given amount of a commodity from party B at a specified price"

# Hard negative only gives partial definition
neg_4 = "A futures contract commits two parties to a buy/sell of the underlying securities, but"

In [49]:
scores = model_msmacro.predict([[query, pos], [query, neg_1], [query, neg_2], [query, neg_3], [query, neg_4]])

scores

array([10.049644 ,  3.0063746,  2.2981606, 10.581794 ,  9.108952 ],
      dtype=float32)

In [51]:
model_qnli.predict([[query, pos], [query, neg_1], [query, neg_2], [query, neg_3], [query, neg_4]])

array([0.9088604 , 0.01259713, 0.00147412, 0.99607915, 0.6586179 ],
      dtype=float32)

In [52]:
model_nli.predict([[query, pos], [query, neg_1], [query, neg_2], [query, neg_3], [query, neg_4]])

array([[-3.2867434, -2.580117 ,  5.8687997],
       [-1.9874078, -2.739047 ,  4.863659 ],
       [-0.5291807, -2.7036226,  3.6301796],
       [-3.3857434, -1.0784333,  4.197326 ],
       [-1.8548764, -3.2716918,  5.377679 ]], dtype=float32)